# 07 - Availability Risk Module

This notebook builds a transparent availability-risk module for the Smart Buy Window Predictor project.

The price-drop models estimate whether waiting is likely to produce a meaningful price drop within a selected prediction window. In V2.1, the system uses separate 7-day, 14-day, and 30-day price-drop models.

However, waiting is not only a price question. It can also be risky if Amazon direct pricing becomes unstable, the price source changes, offer count weakens, or marketplace availability becomes less reliable.

This notebook does not train a supervised stock-availability model because the current dataset does not contain reliable historical daily stock-out labels. Instead, it builds a rule-based availability-risk proxy using historical marketplace signals available at prediction time.

The output is an availability-risk level:

- LOW
- MEDIUM
- HIGH

This module is used as a risk-control layer alongside the price-drop models. The price model estimates whether waiting may produce a saving, while the availability-risk module estimates whether waiting may be risky.

In [2]:
# Cell 2 - Imports and Paths

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

PROJECT_ROOT = Path("/Users/hibaswaidan/Desktop/smart-buy-window-predictor")

FEATURES_PATH = PROJECT_ROOT / "data/processed/features_v2.csv"
OUTPUT_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Features path exists:", FEATURES_PATH.exists())
print("Output directory:", OUTPUT_DIR)

Features path exists: True
Output directory: /Users/hibaswaidan/Desktop/smart-buy-window-predictor/models


In [3]:
# Cell 3 - Load Feature Dataset

df = pd.read_csv(FEATURES_PATH)
df["date"] = pd.to_datetime(df["date"])

print("Dataset shape:", df.shape)
print("Unique ASINs:", df["asin"].nunique())
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())

key_cols = [
    "date",
    "asin",
    "amazon_price",
    "price_source",
    "price_source_amazon",
    "price_source_changed_7d",
    "amazon_price_raw_missing",
    "amazon_price_raw_missing_rolling_14",
    "offer_count",
    "offer_count_lag_7",
    "offer_count_rolling_mean_14",
    "offer_count_trend_14",
    "offer_count_missing_flag",
    "sales_rank",
    "sales_rank_velocity_14",
    "root_category",
    "split"
]

df[key_cols].head()

Dataset shape: (1911211, 66)
Unique ASINs: 922
Date range: 2015-01-31 to 2026-05-18


,date,asin,amazon_price,price_source,price_source_amazon,price_source_changed_7d,amazon_price_raw_missing,amazon_price_raw_missing_rolling_14,offer_count,offer_count_lag_7,offer_count_rolling_mean_14,offer_count_trend_14,offer_count_missing_flag,sales_rank,sales_rank_velocity_14,root_category,split
0,2021-04-15,B00002N6J9,46.2400,AMAZON,1,0.0000,1.0000,0.7857,NaN,3.0000,NaN,NaN,1,7.0000,-0.3000,Tools & Home Improvement,train
1,2021-04-16,B00002N6J9,69.9900,AMAZON,1,0.0000,0.0000,0.7857,2.0000,3.0000,NaN,NaN,0,8.0000,-0.3333,Tools & Home Improvement,train
2,2021-04-17,B00002N6J9,69.9900,AMAZON,1,0.0000,1.0000,0.7143,2.0000,3.0000,NaN,NaN,0,9.0000,-0.1818,Tools & Home Improvement,train
3,2021-04-18,B00002N6J9,69.9900,AMAZON,1,0.0000,1.0000,0.7143,2.0000,3.0000,NaN,NaN,0,11.0000,-1.0000,Tools & Home Improvement,train
4,2021-04-19,B00002N6J9,69.9900,AMAZON,1,0.0000,1.0000,0.7143,2.0000,3.0000,NaN,NaN,0,9.0000,0.0000,Tools & Home Improvement,train


## Availability-Risk Framing

The current dataset does not provide a reliable historical daily stock-out label. Therefore, this notebook does not train a supervised stock-availability model.

Instead, it creates a transparent risk score from historical proxy signals available at prediction time. The goal is to estimate whether waiting may be risky because Amazon direct pricing is unstable, the selected price source is shifting, or the offer market appears weak.

This module is horizon-agnostic. The same availability-risk score can be combined with the 7-day, 14-day, or 30-day price-drop model.

In [4]:
# Cell 5 - Inspect Availability-Related Signals

availability_features = [
    "amazon_price_raw_missing",
    "amazon_price_raw_missing_rolling_14",
    "price_source_amazon",
    "price_source_changed_7d",
    "offer_count",
    "offer_count_lag_7",
    "offer_count_rolling_mean_14",
    "offer_count_trend_14",
    "offer_count_missing_flag",
    "sales_rank_velocity_14"
]

availability_summary = df[availability_features].agg(["count", "mean", "std", "min", "median", "max"]).T
availability_summary["missing_percentage"] = df[availability_features].isna().mean() * 100
availability_summary = availability_summary.round(4)

availability_summary

,count,mean,std,min,median,max,missing_percentage
amazon_price_raw_missing,1911211.0000,0.7800,0.4142,0.0000,1.0000,1.0000,0.0000
amazon_price_raw_missing_rolling_14,1911211.0000,0.7805,0.2502,0.0000,0.8571,1.0000,0.0000
price_source_amazon,1911211.0000,0.6297,0.4829,0.0000,1.0000,1.0000,0.0000
price_source_changed_7d,1911211.0000,0.2476,0.4316,0.0000,0.0000,1.0000,0.0000
offer_count,1323065.0000,13.4998,17.6368,1.0000,8.0000,357.0000,30.7735
offer_count_lag_7,1322027.0000,13.4955,17.6421,1.0000,8.0000,357.0000,30.8278
offer_count_rolling_mean_14,970942.0000,16.7684,19.2017,1.0000,11.1429,341.1429,49.1976
offer_count_trend_14,1140906.0000,-0.0116,4.7691,-134.0000,0.0000,110.0000,40.3046
offer_count_missing_flag,1911211.0000,0.3077,0.4616,0.0000,0.0000,1.0000,0.0000
sales_rank_velocity_14,1801791.0000,1.3285,68.3534,-1.0000,0.0000,24121.7391,5.7252


In [5]:
# Cell 6 - Create Availability-Risk Score

risk_df = df.copy()

# Signal 1: Amazon direct price has been frequently missing recently.
# This is useful, but not enough by itself to mean stock risk because Keepa is event-based.
risk_df["risk_amazon_missing_recent"] = (
    risk_df["amazon_price_raw_missing_rolling_14"] >= 0.85
).astype(int)

# Signal 2: Selected price currently comes from NEW rather than AMAZON.
risk_df["risk_not_amazon_source"] = (
    risk_df["price_source_amazon"] == 0
).astype(int)

# Signal 3: Price source changed recently between AMAZON and NEW.
risk_df["risk_price_source_changed"] = (
    risk_df["price_source_changed_7d"] == 1
).astype(int)

# Signal 4: Offer count is missing.
risk_df["risk_offer_count_missing"] = (
    risk_df["offer_count_missing_flag"] == 1
).astype(int)

# Signal 5: Offer count is very low.
risk_df["risk_low_offer_count"] = (
    risk_df["offer_count"].fillna(9999) <= 2
).astype(int)

# Signal 6: Offer count is declining meaningfully.
risk_df["risk_offer_count_declining"] = (
    risk_df["offer_count_trend_14"].fillna(0) <= -2
).astype(int)

# Weighted risk score.
# Stronger weight is given to direct marketplace weakness signals:
# not Amazon source, very low offer count, and declining offer count.
risk_df["availability_risk_score"] = (
    1 * risk_df["risk_amazon_missing_recent"] +
    2 * risk_df["risk_not_amazon_source"] +
    1 * risk_df["risk_price_source_changed"] +
    1 * risk_df["risk_offer_count_missing"] +
    2 * risk_df["risk_low_offer_count"] +
    2 * risk_df["risk_offer_count_declining"]
)

risk_components = [
    "risk_amazon_missing_recent",
    "risk_not_amazon_source",
    "risk_price_source_changed",
    "risk_offer_count_missing",
    "risk_low_offer_count",
    "risk_offer_count_declining",
    "availability_risk_score"
]

risk_df[risk_components].describe().T.round(4)

,count,mean,std,min,25%,50%,75%,max
risk_amazon_missing_recent,1911211.0000,0.5977,0.4904,0.0000,0.0000,1.0000,1.0000,1.0000
risk_not_amazon_source,1911211.0000,0.3703,0.4829,0.0000,0.0000,0.0000,1.0000,1.0000
risk_price_source_changed,1911211.0000,0.2476,0.4316,0.0000,0.0000,0.0000,0.0000,1.0000
risk_offer_count_missing,1911211.0000,0.3077,0.4616,0.0000,0.0000,0.0000,1.0000,1.0000
risk_low_offer_count,1911211.0000,0.1431,0.3502,0.0000,0.0000,0.0000,0.0000,1.0000
risk_offer_count_declining,1911211.0000,0.1403,0.3473,0.0000,0.0000,0.0000,0.0000,1.0000
availability_risk_score,1911211.0000,2.4607,1.5953,0.0000,1.0000,2.0000,4.0000,8.0000


In [6]:
# Cell 7 - Assign Availability Risk Levels

risk_df["availability_risk_level"] = pd.cut(
    risk_df["availability_risk_score"],
    bins=[-1, 2, 4, 8],
    labels=["LOW", "MEDIUM", "HIGH"]
)

risk_level_summary = (
    risk_df["availability_risk_level"]
    .value_counts()
    .reset_index()
)

risk_level_summary.columns = ["availability_risk_level", "rows"]
risk_level_summary["percentage"] = (
    risk_level_summary["rows"] / len(risk_df) * 100
).round(2)

risk_level_summary

,availability_risk_level,rows,percentage
0,LOW,1021246,53.4300
1,MEDIUM,656519,34.3500
2,HIGH,233446,12.2100


The availability-risk score is intentionally conservative. Most rows are classified as LOW or MEDIUM risk, while only 12.21% are classified as HIGH risk. This avoids over-warning the user based only on weak proxy signals. HIGH risk is reserved for cases where several warning signals appear together, such as unstable Amazon source, low offer count, missing offer data, or declining offer count.

In [7]:
# Cell 8 - Inspect Risk Levels Against Key Marketplace Signals

risk_signal_summary = (
    risk_df
    .groupby("availability_risk_level", observed=True)
    .agg(
        rows=("asin", "count"),
        avg_risk_score=("availability_risk_score", "mean"),
        amazon_source_rate=("price_source_amazon", "mean"),
        avg_amazon_missing_14d=("amazon_price_raw_missing_rolling_14", "mean"),
        avg_offer_count=("offer_count", "mean"),
        offer_count_missing_rate=("offer_count_missing_flag", "mean"),
        low_offer_count_rate=("risk_low_offer_count", "mean"),
        offer_declining_rate=("risk_offer_count_declining", "mean"),
        source_changed_rate=("risk_price_source_changed", "mean"),
        avg_price=("amazon_price", "mean")
    )
    .reset_index()
)

percentage_cols = [
    "amazon_source_rate",
    "avg_amazon_missing_14d",
    "offer_count_missing_rate",
    "low_offer_count_rate",
    "offer_declining_rate",
    "source_changed_rate"
]

risk_signal_summary[percentage_cols] = (
    risk_signal_summary[percentage_cols] * 100
).round(2)

risk_signal_summary["avg_risk_score"] = risk_signal_summary["avg_risk_score"].round(2)
risk_signal_summary["avg_offer_count"] = risk_signal_summary["avg_offer_count"].round(2)
risk_signal_summary["avg_price"] = risk_signal_summary["avg_price"].round(2)

risk_signal_summary

,availability_risk_level,rows,avg_risk_score,amazon_source_rate,avg_amazon_missing_14d,avg_offer_count,offer_count_missing_rate,low_offer_count_rate,offer_declining_rate,source_changed_rate,avg_price
0,LOW,1021246,1.2100,99.4600,67.5900,14.0400,40.9300,6.5000,5.4400,15.4900,77.9000
1,MEDIUM,656519,3.3900,27.2700,89.7900,14.1100,24.4600,14.2100,12.3100,33.7400,86.8600
2,HIGH,233446,5.3000,3.6900,90.8100,10.6900,4.1200,48.7800,56.4400,40.0700,99.4600


## Availability-Risk Signal Interpretation

The risk levels show a clear progression in marketplace weakness.

LOW-risk rows are mostly Amazon-sourced, have relatively stable offer signals, and rarely show low or declining offer count.

HIGH-risk rows are very different: Amazon source rate drops sharply, low offer-count cases become much more common, offer-count decline becomes much more frequent, and price-source switching is higher. This suggests that the rule-based risk score is capturing meaningful marketplace instability rather than assigning risk randomly.

This module should still be interpreted as an availability-risk proxy, not a true stock-out prediction model, because the dataset does not contain reliable historical daily stock labels.

In [8]:
# Cell 9 - Availability Risk by Category

category_risk_summary = (
    risk_df
    .groupby(["root_category", "availability_risk_level"], observed=True)
    .size()
    .reset_index(name="rows")
)

category_totals = (
    risk_df
    .groupby("root_category", observed=True)
    .size()
    .reset_index(name="category_rows")
)

category_risk_summary = category_risk_summary.merge(
    category_totals,
    on="root_category",
    how="left"
)

category_risk_summary["percentage_within_category"] = (
    category_risk_summary["rows"] / category_risk_summary["category_rows"] * 100
).round(2)

category_risk_pivot = category_risk_summary.pivot(
    index="root_category",
    columns="availability_risk_level",
    values="percentage_within_category"
).fillna(0)

category_risk_pivot = category_risk_pivot[["LOW", "MEDIUM", "HIGH"]]
category_risk_pivot

availability_risk_level,LOW,MEDIUM,HIGH
root_category,,,
Appliances,60.0600,28.9300,11.0100
Electronics,40.7300,42.4700,16.8000
Home & Kitchen,53.3800,34.7500,11.8700
Sports & Outdoors,60.5800,29.0800,10.3400
Tools & Home Improvement,55.3900,33.6600,10.9600
Toys & Games,55.6000,32.2800,12.1200


## Category-Level Availability Risk

The category-level risk distribution shows that HIGH availability risk remains a minority class across all product groups. Electronics has the highest HIGH-risk share at 16.80%, while the other categories are mostly between 10% and 12%.

This pattern is reasonable because Electronics may have more seller competition, source switching, and offer instability. The module therefore appears to capture category differences without classifying most products as high risk.

In [9]:
# Cell 10 - Define Price and Availability Combination Policy

def combine_price_and_availability(price_zone, availability_risk_level, horizon=None):
    """
    Combine the price-drop recommendation zone with availability risk.

    price_zone:
        confident_buy, uncertain, confident_wait

    availability_risk_level:
        LOW, MEDIUM, HIGH

    horizon:
        Optional prediction horizon. If horizon is 30 and the system recommends waiting,
        the user-facing label becomes WAIT_AND_TRACK.
    """

    if price_zone == "confident_buy":
        return "BUY_NOW"

    if price_zone == "uncertain":
        if availability_risk_level == "HIGH":
            return "BUY_NOW"
        return "UNCERTAIN"

    if price_zone == "confident_wait":
        if availability_risk_level == "HIGH":
            return "UNCERTAIN"

        if availability_risk_level == "MEDIUM":
            return "WAIT_WITH_CAUTION"

        if horizon == 30:
            return "WAIT_AND_TRACK"

        return "WAIT"

    return "UNCERTAIN"


combination_policy_rows = []

for price_zone in ["confident_buy", "uncertain", "confident_wait"]:
    for availability_risk_level in ["LOW", "MEDIUM", "HIGH"]:
        combination_policy_rows.append({
            "price_zone": price_zone,
            "availability_risk_level": availability_risk_level,
            "7_day_recommendation": combine_price_and_availability(
                price_zone,
                availability_risk_level,
                horizon=7
            ),
            "14_day_recommendation": combine_price_and_availability(
                price_zone,
                availability_risk_level,
                horizon=14
            ),
            "30_day_recommendation": combine_price_and_availability(
                price_zone,
                availability_risk_level,
                horizon=30
            ),
        })

combination_policy_df = pd.DataFrame(combination_policy_rows)

display(combination_policy_df)

,price_zone,availability_risk_level,7_day_recommendation,14_day_recommendation,30_day_recommendation
0,confident_buy,LOW,BUY_NOW,BUY_NOW,BUY_NOW
1,confident_buy,MEDIUM,BUY_NOW,BUY_NOW,BUY_NOW
2,confident_buy,HIGH,BUY_NOW,BUY_NOW,BUY_NOW
3,uncertain,LOW,UNCERTAIN,UNCERTAIN,UNCERTAIN
4,uncertain,MEDIUM,UNCERTAIN,UNCERTAIN,UNCERTAIN
5,uncertain,HIGH,BUY_NOW,BUY_NOW,BUY_NOW
6,confident_wait,LOW,WAIT,WAIT,WAIT_AND_TRACK
7,confident_wait,MEDIUM,WAIT_WITH_CAUTION,WAIT_WITH_CAUTION,WAIT_WITH_CAUTION
8,confident_wait,HIGH,UNCERTAIN,UNCERTAIN,UNCERTAIN


The combination policy separates the learned price signal from the rule-based availability-risk signal.

The price-drop model determines whether the product is in a confident BUY, UNCERTAIN, or confident WAIT zone. The availability-risk module then acts as a safety layer.

HIGH availability risk prevents strong WAIT recommendations because waiting may expose the user to product unavailability or price-source instability. MEDIUM risk does not fully block waiting, but converts it into WAIT WITH CAUTION. For 30-day waiting recommendations, the user-facing action becomes WAIT AND TRACK, because longer waiting windows require active monitoring rather than passive waiting.

In [10]:
# Cell 11 - Save Availability-Risk Configuration

import json

availability_risk_config = {
    "module_type": "rule_based_availability_risk",
    "description": "Horizon-agnostic rule-based availability-risk module using historical/proxy marketplace signals.",
    "risk_score_rules": {
        "risk_amazon_missing_recent": {
            "condition": "amazon_price_raw_missing_rolling_14 >= 0.85",
            "weight": 1
        },
        "risk_not_amazon_source": {
            "condition": "price_source_amazon == 0",
            "weight": 2
        },
        "risk_price_source_changed": {
            "condition": "price_source_changed_7d == 1",
            "weight": 1
        },
        "risk_offer_count_missing": {
            "condition": "offer_count_missing_flag == 1",
            "weight": 1
        },
        "risk_low_offer_count": {
            "condition": "offer_count <= 2",
            "weight": 2
        },
        "risk_offer_count_declining": {
            "condition": "offer_count_trend_14 <= -2",
            "weight": 2
        }
    },
    "risk_level_thresholds": {
        "LOW": "score 0 to 2",
        "MEDIUM": "score 3 to 4",
        "HIGH": "score 5 to 8"
    },
    "important_note": (
        "This is an availability-risk proxy, not a supervised stock-out model. "
        "It is used because the current dataset does not contain reliable historical daily stock-out labels. "
        "The same risk score can be combined with 7-day, 14-day, or 30-day price-drop predictions."
    )
}

CONFIG_PATH = OUTPUT_DIR / "availability_risk_config.json"

with open(CONFIG_PATH, "w") as f:
    json.dump(availability_risk_config, f, indent=4)

print("Saved availability-risk config to:", CONFIG_PATH)

Saved availability-risk config to: /Users/hibaswaidan/Desktop/smart-buy-window-predictor/models/availability_risk_config.json


## Notebook Summary

This notebook created a transparent rule-based availability-risk module for the Smart Buy Window Predictor.

The module does not train a supervised stock-availability model because the current dataset does not contain reliable historical daily stock-out labels. Instead, it uses historical proxy signals related to Amazon source stability, offer-count strength, offer-count decline, offer-count missingness, and recent price-source switching.

The final risk score assigns products into three availability-risk levels:

- LOW
- MEDIUM
- HIGH

The HIGH-risk group shows meaningful marketplace weakness, including lower Amazon-source stability, more frequent price-source switching, more low-offer-count cases, and more frequent offer-count decline.

This module is horizon-agnostic. It can be combined with the 7-day, 14-day, or 30-day price-drop model without changing the risk configuration.

In the final recommendation system, the price-drop model estimates whether waiting may produce a meaningful saving, while the availability-risk module estimates whether waiting may be risky. The combined system can therefore recommend BUY NOW, WAIT, WAIT WITH CAUTION, WAIT AND TRACK, or UNCERTAIN.

The saved configuration file is:

`models/availability_risk_config.json`

This module can later be replaced by a supervised availability model if richer historical Buy Box or stock-out labels become available.